In [ ]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import xarray as xr

import sys
sys.path.append("../src")

from yield_spei_perturbation_simulation import *

## Run
One country/case at one time

In [ ]:
dict_planting_date = {
    "ID": "11/01",
    "KH": "05/01",
    "MM": "05/01",
    "PH": "06/01",
    "TH": "05/01",
    "VN": "05/01" 
}
# growing season set as 10 months from planting to harvest

In [ ]:
country_code = "KH" # select from "ID", "KH", "MM", "PH", "TH", and "VN"

gid_harvest_area_df = pd.read_csv(
    f"../../Results/1a_CAs/HAs_{country_code}.csv"
)
gid_harvest_area_df = gid_harvest_area_df[["GID", "cassava"]]
gid_harvest_area_df.columns = ["GID", "harvest_area_ha"]

observed_yield_df = pd.read_csv(
    f"../../Results/3.1_DetrendedYield/"
    f"DetrendedYield_{country_code}_cassava.csv"
)
observed_yield_df = observed_yield_df[["Year", "DetrendQuadratic"]]
observed_yield_df.columns = ["year", "yield_obs_t_ha"]

gid_harvest_area_selected_df = get_gid_ha_by_coverage(
    gid_harvest_area_df,
    threshold=0.9
)

gdf = gpd.read_file(
    "../../Data/Grid_SEA_0p5deg_processed/admin0_grids_0p5deg.gpkg",
    layer=country_code
)
ds = xr.open_dataset("../../Results/2_spei/spei3_0p5deg.nc")
da = ds["SPEI3"]

simulation_results_df = simulate_yearly_yield(
    gid_harvest_area_selected_df,
    gdf,
    da,
    observed_yield_df,
    beta=0.1,
    planting_date=dict_planting_date[country_code],
    growing_months=10,
    sim_start_time="1991/01/01",
    sim_end_time="2020/12/31",
    n_trials=30,
    yield_threshold=0.01*observed_yield_df["yield_obs_t_ha"].mean(),
)

In [ ]:
grid_yield_output_path = Path(
    "../../Results/3.2_YieldSimulationCassava/"
    "cassava_grid_yield_kh_beta0p1_260721.csv"
)

simulation_results_df = simulation_results_df.round(
    {
        "harvest_area_ha": 2,
        "spei": 4,
        "mean_perturbation": 4,
        "yield_sim_t_ha": 4,
        "scale_factor": 4,
        "yield_scaled_t_ha": 4
    }
)

simulation_results_df.to_csv(grid_yield_output_path, index=False)